<a href="https://colab.research.google.com/github/arun21733/climate-change/blob/main/milestone3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit==1.38.0
!pip install pandas==2.2.2
!pip install numpy==1.26.4
!pip install plotly==5.24.0
!pip install scikit-learn==1.5.1
!pip install joblib==1.4.2
!pip install xgboost==2.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: watchdog
    Found existing installation: watchdog 6.0.0
    Uninstalling watchdog-6.0.0:
      Successfully uninstalled watchdog-6.0.0
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.4
    Uninstalling tenacity-9.1.4:
      Successfully uninstalled tenacity-9.1.4
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully unin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 70.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
ERROR: Operation cancelled by user
^C


In [ ]:
!pip cache purge
!rm -rf /usr/local/lib/python3.12/dist-packages/\~*
!rm -rf /usr/local/lib/python3.12/dist-packages/-*

Files removed: 88


In [ ]:
# ────────────────────────────────────────────────
# 0. Installs & Imports
# ────────────────────────────────────────────────

!pip install -q --upgrade scikit-learn xgboost lightgbm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, r2_score
)
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
%matplotlib inline

print("Imports completed.")

Imports completed.


In [ ]:
!pip install opendatasets
import opendatasets as od
od.download("https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: arun24
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository


100%|██████████| 10.2M/10.2M [00:00<00:00, 1.31GB/s]

In [ ]:
!pip install streamlit==1.38.0
import logging
# Suppress the specific Streamlit warning about no runtime found
logging.getLogger('streamlit.runtime.caching.cache_data_api').setLevel(logging.ERROR)

import pandas as pd
import joblib
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from pathlib import Path
import subprocess
import os


# The entire Streamlit application code, to be written to app.py
streamlit_app_code = '''
import logging
# Suppress the specific Streamlit warning about no runtime found
logging.getLogger('streamlit.runtime.caching.cache_data_api').setLevel(logging.ERROR)

import pandas as pd
import joblib
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from pathlib import Path


# ─── Paths ────────────────────────────────────────
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "processed" / "/content/global-weather-repository/GlobalWeatherRepository.csv"
MODEL_PATH = BASE_DIR / "models" / "best_model.pkl"

@st.cache_data
def load_data():
    if not DATA_PATH.exists():
        st.error(f"Data file not found: {DATA_PATH}")
        st.stop()
    df = pd.read_csv(DATA_PATH)
    return df

@st.cache_resource
def load_model():
    if not MODEL_PATH.exists():
        st.error(f"Model file not found: {MODEL_PATH}")
        st.stop()
    model = joblib.load(MODEL_PATH)
    return model

def create_temp_trend_chart(df):
    if 'Year' not in df.columns or 'Temperature' not in df.columns:
        return None
    fig = px.line(df, x='Year', y='Temperature',
                  title='Global Temperature Trend Over Time',
                  markers=True)
    fig.update_layout(xaxis_title="Year", yaxis_title="Temperature (°C)")
    return fig

def create_co2_emission_bar(df, top_n=15):
    if 'Country' not in df.columns or 'CO2_Emissions' not in df.columns: # Changed 'Year' to 'Country'
        return None
    top_countries = df.groupby('Country')['CO2_Emissions'].mean().nlargest(top_n).reset_index()
    fig = px.bar(top_countries, x='CO2_Emissions', y='Country',
                 title=f'Top {top_n} Countries by Average CO2 Emissions',
                 orientation='h', color='CO2_Emissions')
    fig.update_layout(xaxis_title="CO2 Emissions (metric tons)", yaxis_title="Country")
    return fig

def create_correlation_heatmap(df):
    numeric_df = df.select_dtypes(include=['float64', 'int64'])
    if numeric_df.empty:
        return None
    corr = numeric_df.corr()
    fig = px.imshow(corr, text_auto=True, aspect="auto",
                    color_continuous_scale='RdBu_r',
                    title='Feature Correlation Heatmap')
    return fig

st.set_page_config(page_title="ClimateScope Dashboard", layout="wide")

st.title("🌍 ClimateScope – Climate Change Insights")
st.markdown("Milestone 3 – Interactive Dashboard (Plotly + Streamlit)")

df = load_data()

st.subheader("Quick Overview")
col1, col2, col3 = st.columns(3)
col1.metric("Records", f"{len(df):,}")
col2.metric("Countries", df['Country'].nunique() if 'Country' in df else "N/A")
col3.metric("Years Covered", f"{df['Year'].min()} – {df['Year'].max()}" if 'Year' in df else "N/A")

st.subheader("Key Visuals")

tab1, tab2, tab3 = st.tabs(["Temperature Trend", "CO₂ Emissions", "Correlations"])

with tab1:
    fig_temp = create_temp_trend_chart(df)
    if fig_temp:
        st.plotly_chart(fig_temp, use_container_width=True)
    else:
        st.warning("Temperature trend chart requires 'Year' and 'Temperature' columns.")
with tab2:
    fig_co2 = create_co2_emission_bar(df)
    if fig_co2:
        st.plotly_chart(fig_co2, use_container_width=True)
    else:
        st.warning("CO₂ bar chart requires 'Country' and 'CO2_Emissions' columns.")

with tab3:
    fig_corr = create_correlation_heatmap(df)
    if fig_corr:
        st.plotly_chart(fig_corr, use_container_width=True)
    else:
        st.warning("Correlation heatmap requires numeric features.")

st.markdown(" preconceived thoughts:")
st.caption("Navigate to other pages using the sidebar →")
'''

# Write the Streamlit app code to a file
with open("app.py", "w") as f:
    f.write(streamlit_app_code)

# Install localtunnel if not already installed
try:
    subprocess.run(['npm', 'list', '-g', 'localtunnel'], check=True, capture_output=True)
except subprocess.CalledProcessError:
    print("Installing localtunnel...")
    subprocess.run(['npm', 'install', '-g', 'localtunnel'], check=True)
    print("localtunnel installed.")

# Run the Streamlit app and expose it via localtunnel
# Use subprocess.Popen for non-blocking execution
print("Running Streamlit app...")
subprocess.Popen(["streamlit", "run", "app.py", "&", "npx", "localtunnel", "--port", "8501"])

Running Streamlit app...


<Popen: returncode: None args: ['streamlit', 'run', 'app.py', '&', 'npx', 'l...>

In [ ]:
import streamlit as st
# The functions load_data, create_temp_trend_chart, etc., are already defined in a previous cell (ef4O1aodLHxM)
# and are directly accessible without needing to import from 'utils.helpers'.
# from utils.helpers import load_data, create_temp_trend_chart, create_co2_emission_bar, create_correlation_heatmap

st.set_page_config(page_title="ClimateScope Dashboard", layout="wide")

st.title("🌍 ClimateScope – Climate Change Insights")
st.markdown("Milestone 3 – Interactive Dashboard (Plotly + Streamlit)")

df = load_data() # Corrected: load_data() does not take an argument
st.subheader("Quick Overview")
col1, col2, col3 = st.columns(3)
col1.metric("Records", f"{len(df):,}")
col2.metric("Countries", df['Country'].nunique() if 'Country' in df else "N/A")
col3.metric("Years Covered", f"{df['Year'].min()} – {df['Year'].max()}" if 'Year' in df else "N/A")

st.subheader("Key Visuals")

tab1, tab2, tab3 = st.tabs(["Temperature Trend", "CO₂ Emissions", "Correlations"])

with tab1:
    fig_temp = create_temp_trend_chart(df)
    if fig_temp:
        st.plotly_chart(fig_temp, use_container_width=True)
    else:
        st.warning("Temperature trend chart requires 'Year' and 'Temperature' columns.")
with tab2:
    fig_co2 = create_co2_emission_bar(df)
    if fig_co2:
        st.plotly_chart(fig_co2, use_container_width=True)
    else:
        st.warning("CO₂ bar chart requires 'Country' and 'CO2_Emissions' columns.")
with tab3:
    fig_corr = create_correlation_heatmap(df)
    if fig_corr:
        st.plotly_chart(fig_corr, use_container_width=True)
    else:
        st.warning("Correlation heatmap requires numeric features.")

        st.markdown("---")
st.caption("Navigate to other pages using the sidebar →")





2026-03-13 08:30:11.468 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:30:11.471 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:30:11.473 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:30:11.474 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:30:11.476 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:30:11.477 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:30:11.478 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:30:11.479 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [ ]:
import streamlit as st

st.title("📊 Exploratory Data Analysis Dashboard")

df = load_data()
with st.sidebar:
    st.header("Filters")
    if 'Country' in df.columns:
        selected_countries = st.multiselect("Select Countries", options=sorted(df['Country'].unique()), default=[])
    if 'Year' in df.columns:
        year_range = st.slider("Year Range", int(df['Year'].min()), int(df['Year'].max()), (int(df['Year'].min()), int(df['Year'].max())))

    filtered_df = df.copy()
    if 'Country' in df.columns and selected_countries:
        filtered_df = filtered_df[filtered_df['Country'].isin(selected_countries)]
    if 'Year' in df.columns:
        filtered_df = filtered_df[filtered_df['Year'].between(year_range[0], year_range[1])]
st.subheader("Filtered Dataset Preview")
st.dataframe(filtered_df.head(10))

st.subheader("Distribution of Key Variables")
cols = st.columns(2)
with cols[0]:
    if 'Temperature' in filtered_df.columns:
        fig_hist = px.histogram(filtered_df, x='Temperature', title="Temperature Distribution", nbins=40)
        st.plotly_chart(fig_hist, use_container_width=True)

with cols[1]:
    if 'CO2_Emissions' in filtered_df.columns:
        fig_box = px.box(filtered_df, x='CO2_Emissions', title="CO₂ Emissions Box Plot")
        st.plotly_chart(fig_box, use_container_width=True)

2026-03-13 08:32:39.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:32:39.721 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:32:39.731 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:32:39.739 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:32:39.755 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:32:39.934 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:32:39.935 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 08:32:39.937 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
import streamlit as st
import pandas as pd
# The functions load_model, load_data were defined in a previous cell (ef4O1aodLHxM)
# and are directly accessible without needing to import from 'utils.helpers'.
# from utils.helpers import load_model, load_data

st.title("🔮 Climate Impact Predictor")

model = load_model()
df_sample = load_data().sample(1000)   # for getting realistic input ranges

st.subheader("Enter Features for Prediction")

# Example input fields – CHANGE according to your actual features/model
col1, col2 = st.columns(2)

with col1:
    co2 = st.number_input("CO₂ Emissions (metric tons)", min_value=0.0, max_value=50.0, value=5.0, step=0.1)
    temp_anomaly = st.number_input("Temperature Anomaly (°C)", min_value=-2.0, max_value=3.0, value=0.8, step=0.1)

with col2:
    year = st.number_input("Year", min_value=1990, max_value=2050, value=2030, step=1)
    population = st.number_input("Population (millions)", min_value=1.0, max_value=2000.0, value=500.0, step=10.0)
# Add more inputs depending on your model features

if st.button("Predict", type="primary"):
    input_data = pd.DataFrame({
        'CO2_Emissions': [co2],
        'Temperature_Anomaly': [temp_anomaly],
        'Year': [year],
        'Population': [population],
        # add all other features your model expects – use 0 or median if missing
    })

    # Make sure columns match training data (very important!)
    # You can do: input_data = input_data.reindex(columns=model.feature_names_in_, fill_value=0)

    try:
        prediction = model.predict(input_data)[0]
        st.success(f"**Predicted Value**: {prediction:.4f}")

        if hasattr(model, "predict_proba"):
            prob = model.predict_proba(input_data)[0]
            st.info(f"Class probabilities: {dict(zip(model.classes_, prob.round(3)))}")
    except Exception as e:
        st.error(f"Prediction failed: {str(e)}")
        st.info("Make sure input features match the model's training columns.")


ModuleNotFoundError: No module named 'streamlit'

In [ ]:
# ────────────────────────────────────────────────
# 1. Prepare Data for Model Training
# ────────────────────────────────────────────────

# Load the dataset into df
df = load_data()

# Clean column names by stripping whitespace
df.columns = df.columns.str.strip()

# Convert 'last_updated' to datetime and extract 'year'
df['last_updated'] = pd.to_datetime(df['last_updated'])
df['year'] = df['last_updated'].dt.year

# Create a synthetic 'co2_emissions_tons_per_person' column
# (as this column is not present in the GlobalWeatherRepository.csv)
df['co2_emissions_tons_per_person'] = df['temperature_celsius'] * 0.1 + df['humidity'] * 0.05 + np.random.rand(len(df)) * 2 # Example synthetic data

# Drop rows with any missing values for simplicity in this example
# Using available relevant numerical columns for subset
df_model = df.dropna(subset=['co2_emissions_tons_per_person', 'temperature_celsius', 'year', 'humidity', 'wind_kph', 'pressure_mb']).copy()

# Define target variable: let's classify if CO2 emissions are 'high' or 'low'
# This is a simplification; in a real scenario, you might predict the value or use more sophisticated features.
# For demonstration, we'll create a binary target based on the median co2_emissions_tons_per_person.
median_co2 = df_model['co2_emissions_tons_per_person'].median()
df_model['High_CO2_Emissions'] = (df_model['co2_emissions_tons_per_person'] > median_co2).astype(int)

# Select features for the model - using available relevant numerical columns
features = ['temperature_celsius', 'year', 'humidity', 'wind_kph', 'pressure_mb']
target = 'High_CO2_Emissions'

X = df_model[features]
y = df_model[target]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# ────────────────────────────────────────────────
# 2. Train and Save Model
# ────────────────────────────────────────────────

# Initialize and train an XGBoost Classifier
model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

# Evaluate the model (optional, but good practice)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Create the directory if it doesn't exist
model_dir = Path('models')
model_dir.mkdir(parents=True, exist_ok=True)

# Define the model path as expected by the Streamlit app
MODEL_PATH = model_dir / "best_model.pkl"

# Save the trained model
joblib.dump(model, MODEL_PATH)

print(f"Model trained and saved to: {MODEL_PATH}")

NameError: name 'load_data' is not defined

In [ ]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.81.245.101:8501

